# SCIMAP automatic gating after RAPIDS DAPI-QC filtering

This notebook receives the filtered and clustered AnnData produced by the RAPIDS notebook:

```text
outputs/03_RAPIDS_clustering/<sample>/clustering/rapids_clustered.h5ad
```

The notebook keeps the same core SCIMAP logic tested manually:

```python
adata_lineage = adata[:, lineage_markers].copy()

adata_lineage = sm.pp.rescale(
    adata_lineage,
    gate=None,
    log=True,
    imageid="imageid",
    method="all",
    gmm_components=3,
    verbose=True,
    random_state=0,
)
```

After SCIMAP rescaling, the positivity call is:

```python
scimap_rescaled_value > 0.5
```

Population names are reported with spaces:

```text
CD3+ CD4+ Foxp3+
```

not pipe-separated combinations.

Outputs are saved as final SCIMAP-gated AnnData, PDF tables, PDF plots, and the executed notebook.


In [ ]:

# ==============================================================================
# PARAMETERS
# ==============================================================================
# Placeholder values.
# Nextflow + Papermill override these at runtime.
# ==============================================================================

SAMPLE_INDEX = 1
SAMPLE_ID = "sample"
SAMPLE_DIR_NAME = "001_sample"

INPUT_PATH = "rapids_clustered.h5ad"
OUTPUT_DIR = "scimap_output"

# SCIMAP automatic gating parameters
SCIMAP_GMM_COMPONENTS = 3
SCIMAP_LOG = True
SCIMAP_METHOD = "all"
SCIMAP_RANDOM_STATE = 0
SCIMAP_POSITIVITY_THRESHOLD = 0.5

# Reporting parameters
TOP_N_POPULATIONS = 30
TOP_N_SPATIAL_POPULATIONS = 15
TOP_N_PER_PANEL = 15

# Safety check:
# the input should already be the DAPI-QC-filtered RAPIDS AnnData
REQUIRE_DAPI_QC_FILTERED = True


In [ ]:

# ==============================================================================
# SECTION 1: IMPORTS AND OUTPUT DIRECTORIES
# ==============================================================================
# Keep imports simple.
# Do not add RAPIDS packages here.
# ==============================================================================

from pathlib import Path
import math

import scimap as sm
import anndata as ad
import pandas as pd
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

from IPython.display import display


# ------------------------------------------------------------------------------
# Optional Great Tables display.
# Saved PDF tables use matplotlib to avoid extra rendering dependencies.
# ------------------------------------------------------------------------------

try:
    from great_tables import GT
    GREAT_TABLES_AVAILABLE = True
except Exception as e:
    GT = None
    GREAT_TABLES_AVAILABLE = False
    print("WARNING: great_tables is not available. Tables will display as pandas DataFrames.")
    print("Import error:", repr(e))


def display_table(df):
    """
    Display a table with Great Tables when available.
    Fallback to pandas display if Great Tables is not installed.
    """
    if GREAT_TABLES_AVAILABLE:
        display(GT(df))
    else:
        display(df)


# ------------------------------------------------------------------------------
# Output directories
# ------------------------------------------------------------------------------

OUTPUT_DIR = Path(OUTPUT_DIR)

TABLE_DIR = OUTPUT_DIR / "tables"
PLOT_DIR = OUTPUT_DIR / "plots"
CLUSTERING_DIR = OUTPUT_DIR / "clustering"
REPORT_DIR = OUTPUT_DIR / "report"

for directory in [TABLE_DIR, PLOT_DIR, CLUSTERING_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


print("SCIMAP automatic gating notebook initialized")
print(f"SAMPLE_INDEX: {SAMPLE_INDEX}")
print(f"SAMPLE_ID: {SAMPLE_ID}")
print(f"SAMPLE_DIR_NAME: {SAMPLE_DIR_NAME}")
print(f"INPUT_PATH: {INPUT_PATH}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"SCIMAP_GMM_COMPONENTS: {SCIMAP_GMM_COMPONENTS}")
print(f"SCIMAP_LOG: {SCIMAP_LOG}")
print(f"SCIMAP_METHOD: {SCIMAP_METHOD}")
print(f"SCIMAP_RANDOM_STATE: {SCIMAP_RANDOM_STATE}")
print(f"SCIMAP_POSITIVITY_THRESHOLD: {SCIMAP_POSITIVITY_THRESHOLD}")


In [ ]:

# ==============================================================================
# SECTION 2: LOAD RAPIDS ANNDATA AND STANDARDIZE METADATA
# ==============================================================================
# The SCIMAP environment can be sensitive to AnnData versions.
# If your tested notebook has a more specific H5AD import workaround,
# replace only the inside of read_h5ad_compat(), not the rest of the workflow.
# ==============================================================================

import h5py

def read_h5ad_compat(path):
    path = str(path)

    # Compatibility fix for RAPIDS/Scanpy AnnData output.
    # Older anndata inside the SCIMAP environment can fail on /uns/log1p.
    # This removes only that metadata group before reading.
    with h5py.File(path, "a") as f:
        if "uns/log1p" in f:
            del f["uns/log1p"]
            print("Patched: removed /uns/log1p from RAPIDS AnnData")

    return ad.read_h5ad(path)


adata = read_h5ad_compat(INPUT_PATH)

print("Loaded AnnData:")
print(adata)

print("\nMarkers:")
print(list(adata.var_names))


# ------------------------------------------------------------------------------
# Check / enforce that we work on RAPIDS DAPI-QC-passing cells
# ------------------------------------------------------------------------------

dapi_qc_info = adata.uns.get("dapi_qc", {})

print("\nDAPI QC info from adata.uns['dapi_qc']:")
print(dapi_qc_info)

if REQUIRE_DAPI_QC_FILTERED:
    is_filtered = bool(dapi_qc_info.get("final_adata_is_dapi_qc_filtered", False))

    if is_filtered:
        print("Input AnnData is marked as DAPI-QC filtered.")

    elif "qc_keep_for_analysis" in adata.obs.columns:
        print(
            "WARNING: AnnData is not marked as fully DAPI-QC filtered, "
            "but qc_keep_for_analysis exists. Filtering to QC-passing cells now."
        )
        adata = adata[adata.obs["qc_keep_for_analysis"].astype(bool).values].copy()
        adata.uns.setdefault("dapi_qc", {})
        adata.uns["dapi_qc"]["final_adata_is_dapi_qc_filtered"] = True

    else:
        raise ValueError(
            "This SCIMAP notebook is configured to run only on DAPI-QC-filtered cells. "
            "Expected adata.uns['dapi_qc']['final_adata_is_dapi_qc_filtered'] == True, "
            "or adata.obs['qc_keep_for_analysis'] to be available."
        )


# ------------------------------------------------------------------------------
# Standardize spatial coordinates
# ------------------------------------------------------------------------------

if "spatial" in adata.obsm:
    coords = adata.obsm["spatial"]

    if isinstance(coords, pd.DataFrame):
        coords_array = coords.iloc[:, :2].values
    else:
        coords_array = np.asarray(coords)[:, :2]

    adata.obsm["spatial"] = coords_array
    adata.obs["X_centroid"] = coords_array[:, 0]
    adata.obs["Y_centroid"] = coords_array[:, 1]

elif {"X_centroid", "Y_centroid"}.issubset(adata.obs.columns):
    adata.obsm["spatial"] = adata.obs[["X_centroid", "Y_centroid"]].values

else:
    raise ValueError(
        "No spatial coordinates found. Expected adata.obsm['spatial'] "
        "or X_centroid/Y_centroid in adata.obs."
    )


# ------------------------------------------------------------------------------
# Ensure SCIMAP imageid column exists
# ------------------------------------------------------------------------------

if "imageid" not in adata.obs.columns:
    if "slide" in adata.obs.columns:
        adata.obs["imageid"] = adata.obs["slide"].astype(str).values
    elif "sample_id" in adata.obs.columns:
        adata.obs["imageid"] = adata.obs["sample_id"].astype(str).values
    else:
        adata.obs["imageid"] = SAMPLE_ID

adata.obs["sample_id"] = SAMPLE_ID
adata.obs["sample_index"] = SAMPLE_INDEX
adata.obs["sample_dir_name"] = SAMPLE_DIR_NAME


# Snapshot used for final session-summary table
initial_obs = list(adata.obs.columns)
initial_obsm = list(adata.obsm.keys())
initial_layers = list(adata.layers.keys())
initial_uns = list(adata.uns.keys())


print("\nSCIMAP imageid counts:")
display(adata.obs["imageid"].value_counts().head())


In [ ]:

# ==============================================================================
# SECTION 3: LINEAGE MARKER SELECTION
# ==============================================================================
# Preferred:
#   Use lineage markers saved by the RAPIDS notebook in adata.uns["lineage_markers"].
#
# Fallback:
#   Exclude DAPI and obvious composite/technical channels.
# ==============================================================================

def is_likely_technical_channel(marker_name: str) -> bool:
    s = str(marker_name).strip().lower()

    technical_terms = {
        "dapi",
        "composite_mip",
    }

    return any(term in s for term in technical_terms)


if "lineage_markers" in adata.uns and len(adata.uns["lineage_markers"]) > 0:
    lineage_markers = [
        str(m)
        for m in adata.uns["lineage_markers"]
        if str(m) in adata.var_names
    ]
    print("Using lineage markers from adata.uns['lineage_markers'].")

else:
    lineage_markers = [
        str(m)
        for m in adata.var_names
        if not is_likely_technical_channel(str(m))
    ]
    print("Using fallback lineage-marker selection by excluding technical channels.")

if len(lineage_markers) == 0:
    raise ValueError("No lineage markers found for SCIMAP automatic gating.")

technical_channels = [
    str(m)
    for m in adata.var_names
    if str(m) not in lineage_markers
]

adata.uns["scimap_lineage_markers"] = lineage_markers
adata.uns["scimap_technical_channels"] = technical_channels

marker_table = pd.DataFrame({
    "marker": list(adata.var_names),
    "used_for_scimap_gating": [m in lineage_markers for m in adata.var_names],
    "technical_or_excluded": [m not in lineage_markers for m in adata.var_names],
})

print("Lineage markers used for SCIMAP automatic gating:")
print(lineage_markers)

print("\nTechnical/excluded markers:")
print(technical_channels)

display_table(marker_table)


In [ ]:

# ==============================================================================
# SECTION 4: PDF
# ==============================================================================
# Tables are displayed with Great Tables when available.
# Tables are saved as PDF with matplotlib to avoid extra rendering dependencies.
# ==============================================================================

def save_table_pdf(df, path, title=None, max_rows=None, dpi=300):
    """
    Save a pandas DataFrame as a PDF table.

    Note:
    Matplotlib PDF output is vector-based for text/table objects.
    dpi is still provided for consistency with the rest of the reporting.
    """

    df_to_show = df.copy()

    if max_rows is not None:
        df_to_show = df_to_show.head(max_rows).copy()

    for col in df_to_show.columns:
        if pd.api.types.is_float_dtype(df_to_show[col]):
            df_to_show[col] = df_to_show[col].round(4)

    n_rows, n_cols = df_to_show.shape

    fig_width = max(8, n_cols * 1.6)
    fig_height = max(2.8, n_rows * 0.35 + 1.5)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")

    if title is not None:
        ax.set_title(title, fontsize=12, pad=12)

    table = ax.table(
        cellText=df_to_show.astype(str).values,
        colLabels=df_to_show.columns,
        loc="center",
        cellLoc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.25)

    plt.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


save_table_pdf(
    marker_table,
    TABLE_DIR / "marker_table.pdf",
    title=f"{SAMPLE_ID} - SCIMAP marker table",
    dpi=300,
)


In [ ]:

# ==============================================================================
# SECTION 5: SCIMAP AUTOMATIC GATING
# ==============================================================================
# This is the core logic from the tested SCIMAP notebook.
# Do not change unless SCIMAP itself fails.
# ==============================================================================

adata_lineage = adata[:, lineage_markers].copy()

# SCIMAP needs imageid in obs.
adata_lineage.obs["imageid"] = adata.obs["imageid"].astype(str).values

adata_lineage = sm.pp.rescale(
    adata_lineage,
    gate=None,
    log=SCIMAP_LOG,
    imageid="imageid",
    method=SCIMAP_METHOD,
    gmm_components=SCIMAP_GMM_COMPONENTS,
    verbose=True,
    random_state=SCIMAP_RANDOM_STATE,
)

scaled_matrix = adata_lineage.X.copy()

if sp.issparse(scaled_matrix):
    scaled_matrix = scaled_matrix.toarray()

scaled_matrix = np.asarray(scaled_matrix, dtype=np.float32)

positive_matrix = scaled_matrix > SCIMAP_POSITIVITY_THRESHOLD

print("SCIMAP rescaled matrix shape:")
print(scaled_matrix.shape)

print("SCIMAP positive matrix shape:")
print(positive_matrix.shape)


# ------------------------------------------------------------------------------
# Save rescaled values and positivity calls inside the final AnnData
# ------------------------------------------------------------------------------
# Stored as arrays for safer H5AD writing across AnnData versions.
# Marker order is stored in adata.uns["scimap_lineage_markers"].
# ------------------------------------------------------------------------------

adata.obsm["scimap_rescaled_lineage"] = scaled_matrix
adata.obsm["scimap_positive_lineage"] = positive_matrix.astype(np.uint8)

adata.uns["scimap_gating_parameters"] = {
    "gate": None,
    "log": bool(SCIMAP_LOG),
    "imageid": "imageid",
    "method": str(SCIMAP_METHOD),
    "gmm_components": int(SCIMAP_GMM_COMPONENTS),
    "random_state": int(SCIMAP_RANDOM_STATE),
    "positivity_threshold_after_rescale": float(SCIMAP_POSITIVITY_THRESHOLD),
    "positive_rule": f"scimap_rescaled_value > {SCIMAP_POSITIVITY_THRESHOLD}",
    "lineage_markers": list(lineage_markers),
    "input_was_dapi_qc_filtered": bool(
        adata.uns.get("dapi_qc", {}).get("final_adata_is_dapi_qc_filtered", False)
    ),
}

scimap_gate_summary = pd.DataFrame({
    "marker": lineage_markers,
    "rescaled_gate_value": SCIMAP_POSITIVITY_THRESHOLD,
    "positive_rule": f"scimap_rescaled_value > {SCIMAP_POSITIVITY_THRESHOLD}",
    "scimap_gate_argument": "gate=None",
    "automatic_gate_method": f"GMM, gmm_components={SCIMAP_GMM_COMPONENTS}",
})

adata.uns["scimap_gate_summary"] = scimap_gate_summary

display_table(scimap_gate_summary)

save_table_pdf(
    scimap_gate_summary,
    TABLE_DIR / "scimap_gate_summary.pdf",
    title=f"{SAMPLE_ID} - SCIMAP automatic gate summary",
    dpi=300,
)


In [ ]:

# ==============================================================================
# SECTION 6: MARKER POSITIVITY SUMMARY
# ==============================================================================

marker_positive_summary = pd.DataFrame({
    "marker": lineage_markers,
    "n_positive_cells": positive_matrix.sum(axis=0).astype(int),
})

marker_positive_summary["fraction_positive_cells"] = (
    marker_positive_summary["n_positive_cells"] / adata.n_obs
)

marker_positive_summary = marker_positive_summary.sort_values(
    "n_positive_cells",
    ascending=False,
).reset_index(drop=True)

adata.uns["scimap_marker_positive_summary"] = marker_positive_summary

display_table(marker_positive_summary)

save_table_pdf(
    marker_positive_summary,
    TABLE_DIR / "scimap_marker_positive_summary.pdf",
    title=f"{SAMPLE_ID} - SCIMAP marker positivity summary",
    dpi=300,
)


In [ ]:

# ==============================================================================
# SECTION 7: POPULATION CALLS
# ==============================================================================
# Population labels use spaces:
#
# e.g CD3+ CD4+ Foxp3+
# ==============================================================================

positive_df = pd.DataFrame(
    positive_matrix,
    index=adata.obs_names,
    columns=lineage_markers,
)

population_labels = []

for _, row in positive_df.iterrows():
    positive_markers = [
        f"{marker}+"
        for marker in lineage_markers
        if bool(row[marker])
    ]

    if len(positive_markers) == 0:
        population_labels.append("all_negative")
    else:
        population_labels.append(" ".join(positive_markers))

adata.obs["scimap_population"] = pd.Categorical(population_labels)
adata.obs["n_positive_lineage_markers"] = positive_df.sum(axis=1).astype(np.uint8).values

population_counts = (
    adata.obs["scimap_population"]
    .astype(str)
    .value_counts()
    .rename_axis("scimap_population")
    .reset_index(name="n_cells")
)

population_counts["fraction_cells"] = population_counts["n_cells"] / adata.n_obs

population_counts["n_positive_markers"] = population_counts[
    "scimap_population"
].apply(
    lambda x: 0 if x == "all_negative" else len(str(x).split(" "))
)

adata.uns["scimap_population_counts"] = population_counts

print("Top SCIMAP populations:")
display_table(population_counts.head(TOP_N_POPULATIONS))

save_table_pdf(
    population_counts.head(TOP_N_POPULATIONS),
    TABLE_DIR / "scimap_population_counts_top30.pdf",
    title=f"{SAMPLE_ID} - top {TOP_N_POPULATIONS} SCIMAP populations",
    dpi=300,
)


In [ ]:
# ==============================================================================
# SECTION 8: POPULATION COUNT PLOTS
# ==============================================================================
# Outputs:
#   1. top population count plot
#   2. multi-page PDF:
#        Distribution of Cell Groups Defined by Positive Lineage Marker
#      one page per number of positive markers
#   3. PNG images for each page of the multi-page PDF
# ==============================================================================

from matplotlib.backends.backend_pdf import PdfPages
import textwrap


def wrap_label(x, width=55):
    return "\n".join(textwrap.wrap(str(x), width=width))


# ------------------------------------------------------------------------------
# Plot 1: top SCIMAP populations
# ------------------------------------------------------------------------------

top_populations = (
    adata.obs["scimap_population"]
    .astype(str)
    .value_counts()
    .head(TOP_N_POPULATIONS)
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(11, 9))

ax.barh(
    [wrap_label(x, width=70) for x in top_populations.index],
    top_populations.values,
    edgecolor="black",
    linewidth=0.4,
)

ax.set_xlabel("Number of cells")
ax.set_ylabel("SCIMAP population")
ax.set_title(f"{SAMPLE_ID} - top {TOP_N_POPULATIONS} SCIMAP populations")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

fig.savefig(
    PLOT_DIR / "scimap_population_counts_top30.pdf",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    PLOT_DIR / "scimap_population_counts_top30.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------------------------
# Plot 2: Distribution of Cell Groups Defined by Positive Lineage Marker
# ------------------------------------------------------------------------------
# This replaces the unreadable one-page multi-panel PDF.
# Same information, but one readable page per number of positive markers.
# ------------------------------------------------------------------------------

dist_pdf = PLOT_DIR / "scimap_populations_by_n_positive_markers.pdf"
dist_png_dir = PLOT_DIR / "scimap_populations_by_n_positive_markers_pages"
dist_png_dir.mkdir(parents=True, exist_ok=True)

n_values = sorted(population_counts["n_positive_markers"].unique())

with PdfPages(dist_pdf) as pdf:
    for n in n_values:
        sub = population_counts[
            population_counts["n_positive_markers"] == n
        ].copy()

        sub = sub.sort_values("n_cells", ascending=False)

        if TOP_N_PER_PANEL is not None:
            sub = sub.head(TOP_N_PER_PANEL)

        sub = sub.sort_values("n_cells", ascending=True)

        labels = [wrap_label(x, width=70) for x in sub["scimap_population"]]

        fig_h = max(5, 0.65 * len(sub) + 2.5)
        fig_w = 13

        fig, ax = plt.subplots(figsize=(fig_w, fig_h))

        ax.barh(
            labels,
            sub["n_cells"],
            edgecolor="black",
            linewidth=0.4,
        )

        ax.set_xlabel("Number of cells")
        ax.set_ylabel("Cell group defined by positive lineage marker")
        ax.set_title(
            "Distribution of Cell Groups Defined by Positive Lineage Marker\n"
            f"{SAMPLE_ID} - {n} positive marker{'s' if n != 1 else ''}"
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()

        pdf.savefig(fig, bbox_inches="tight")

        fig.savefig(
            dist_png_dir / f"positive_lineage_marker_groups_{int(n):02d}.png",
            dpi=300,
            bbox_inches="tight",
        )

        plt.close(fig)

print("Saved readable multi-page PDF:")
print(dist_pdf)

print("Saved PNG pages to:")
print(dist_png_dir)

In [ ]:
# ==============================================================================
# SECTION 9: SPATIAL PLOTS
# ==============================================================================
# Outputs:
#   1. Spatial Distribution by Number of Positive Lineage Markers
#   2. Top 15 positive lineage marker groups — SCIMAP style
#   3. Top 15 positive lineage marker groups — matplotlib style
# ==============================================================================

x_col = "X_centroid"
y_col = "Y_centroid"


# ------------------------------------------------------------------------------
# Plot 1: Spatial Distribution by Number of Positive Lineage Markers
# ------------------------------------------------------------------------------

group_col = "n_positive_lineage_markers"
n_values = sorted(adata.obs[group_col].unique())

ncols = 3
n_panels = len(n_values)
nrows = math.ceil(n_panels / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(18, 5 * nrows),
)

axes = np.asarray(axes).flatten()

for i, n in enumerate(n_values):
    ax = axes[i]

    mask = adata.obs[group_col] == n

    ax.scatter(
        adata.obs.loc[mask, x_col],
        adata.obs.loc[mask, y_col],
        s=0.15,
        alpha=0.65,
        linewidths=0,
        rasterized=True,
    )

    ax.set_title(
        f"{n} positive marker{'s' if n != 1 else ''}\n"
        f"n = {int(mask.sum()):,}",
        fontsize=11,
    )

    ax.set_xlabel("X centroid")
    ax.set_ylabel("Y centroid")
    ax.invert_yaxis()
    ax.set_aspect("equal")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle(
    f"{SAMPLE_ID} - Spatial Distribution by Number of Positive Lineage Markers",
    fontsize=16,
    y=1.02,
)

plt.tight_layout()

fig.savefig(
    PLOT_DIR / "spatial_by_n_positive_lineage_markers.pdf",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    PLOT_DIR / "spatial_by_n_positive_lineage_markers.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------------------------
# Prepare top 15 positive lineage marker groups
# ------------------------------------------------------------------------------

top_groups = (
    adata.obs["scimap_population"]
    .astype(str)
    .value_counts()
    .head(TOP_N_SPATIAL_POPULATIONS)
    .index
    .tolist()
)

adata.obs["plot_groups_top15"] = adata.obs["scimap_population"].astype(str).where(
    adata.obs["scimap_population"].astype(str).isin(top_groups),
    other="other",
)

adata.obs["plot_groups_top15"] = pd.Categorical(
    adata.obs["plot_groups_top15"],
    categories=top_groups + ["other"],
)


# ------------------------------------------------------------------------------
# Plot 2: Matplotlib spatial style
# ------------------------------------------------------------------------------

group_col = "plot_groups_top15"

groups = [g for g in adata.obs[group_col].cat.categories if g != "other"]

groups = sorted(
    groups,
    key=lambda g: adata.obs[group_col].value_counts().get(g, 0),
    reverse=True,
)

cmap = plt.get_cmap("tab20")

color_map = {
    group: cmap(i % 20)
    for i, group in enumerate(groups)
}

color_map["other"] = "lightgrey"

fig, ax = plt.subplots(figsize=(9, 8))

# Plot other cells first
mask_other = adata.obs[group_col] == "other"

ax.scatter(
    adata.obs.loc[mask_other, "X_centroid"],
    adata.obs.loc[mask_other, "Y_centroid"],
    s=0.3,
    c=color_map["other"],
    alpha=0.25,
    linewidths=0,
    label="other",
    rasterized=True,
)

# Plot top groups on top
for group in groups:
    mask = adata.obs[group_col] == group

    ax.scatter(
        adata.obs.loc[mask, "X_centroid"],
        adata.obs.loc[mask, "Y_centroid"],
        s=1,
        c=[color_map[group]],
        alpha=0.8,
        linewidths=0,
        label=group,
        rasterized=True,
    )

ax.invert_yaxis()
ax.set_aspect("equal")

ax.set_xlabel("X centroid")
ax.set_ylabel("Y centroid")
ax.set_title(
    f"{SAMPLE_ID} - Spatial Distribution of the Top "
    f"{TOP_N_SPATIAL_POPULATIONS} Positive Lineage Marker Groups"
)

ax.legend(
    title="Positive group",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    markerscale=5,
    fontsize=8,
)

plt.tight_layout()

# Keep this expected filename for Nextflow validation.
fig.savefig(
    PLOT_DIR / "spatial_top_scimap_populations.pdf",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    PLOT_DIR / "spatial_top_scimap_populations.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:

# ==============================================================================
# SECTION 10: SAVE FINAL SCIMAP-GATED ANNDATA AND SESSION SUMMARY
# ==============================================================================

adata.uns["scimap_autogating"] = {
    "sample_index": int(SAMPLE_INDEX),
    "sample_id": str(SAMPLE_ID),
    "sample_dir_name": str(SAMPLE_DIR_NAME),
    "input_path": str(INPUT_PATH),
    "n_cells": int(adata.n_obs),
    "lineage_markers": list(lineage_markers),
    "technical_channels": list(technical_channels),
    "population_column": "scimap_population",
    "positivity_matrix_obsm": "scimap_positive_lineage",
    "rescaled_matrix_obsm": "scimap_rescaled_lineage",
    "scimap_positivity_threshold": float(SCIMAP_POSITIVITY_THRESHOLD),
}

FINAL_ADATA_PATH = CLUSTERING_DIR / "scimap_gated.h5ad"
adata.write_h5ad(FINAL_ADATA_PATH)

print("Final SCIMAP-gated AnnData saved:")
print(FINAL_ADATA_PATH)
print(adata)


# ------------------------------------------------------------------------------
# Session summary
# ------------------------------------------------------------------------------

new_obs = [k for k in adata.obs.columns if k not in initial_obs]
new_obsm = [k for k in adata.obsm.keys() if k not in initial_obsm]
new_layers = [k for k in adata.layers.keys() if k not in initial_layers]
new_uns = [k for k in adata.uns.keys() if k not in initial_uns]

rows = []

for k in new_layers:
    rows.append({
        "Slot": ".layers",
        "Variable Name": k,
        "Format / Shape": f"Matrix {adata.layers[k].shape}",
    })

for k in new_obsm:
    rows.append({
        "Slot": ".obsm",
        "Variable Name": k,
        "Format / Shape": f"Array/DataFrame {adata.obsm[k].shape}",
    })

for k in new_obs:
    rows.append({
        "Slot": ".obs",
        "Variable Name": k,
        "Format / Shape": f"Series ({adata.n_obs},)",
    })

for k in new_uns:
    rows.append({
        "Slot": ".uns",
        "Variable Name": k,
        "Format / Shape": type(adata.uns[k]).__name__,
    })

session_summary = pd.DataFrame(rows)

save_table_pdf(
    session_summary,
    REPORT_DIR / "session_summary.pdf",
    title=f"{SAMPLE_ID} - SCIMAP variables generated",
    dpi=300,
)

display_table(session_summary)

print("SCIMAP automatic gating notebook complete.")
